<a href="https://colab.research.google.com/github/attabeezy/crop-guard/blob/main/notebooks/05_export_int8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropGuard — full INT8 export and deployment parity

This notebook converts the selected Keras checkpoint into a fully integer-quantized TensorFlow Lite model using real training images as the representative dataset. It then evaluates the INT8 artifact across the complete held-out test set and compares it with the locked float-model predictions.

The confidence temperature and threshold remain external post-processing parameters. This keeps the neural network fully quantized while preserving the exact evaluated uncertainty policy.

## 1. Dependencies and deterministic setup

A CPU runtime is sufficient, although conversion and full parity evaluation may take several minutes.

In [ ]:
%pip install -q kagglehub pandas scikit-learn pillow tqdm

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 2026
IMAGE_SIZE = 224
REPRESENTATIVE_IMAGES = 500
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)

## 2. Download and verify pinned model, manifests, and evaluation decisions

In [ ]:
import hashlib
import io
import json
import shutil
import urllib.request
import zipfile

AUDIT_COMMIT = "4c8b366df4c251d6424258c3cb74a58f2289ee16"
AUDIT_SHA256 = "1f69013b0a0fc58cba5dccab37d50e8767902580f443ac13daa3fba885e3269c"
MODEL_COMMIT = "08d86ed7807075b195a43acb80e11ef738ee4b28"
MODEL_SHA256 = "14c7ed601c3bebd64238a1a3375c1653bc6017009df645a22878241525e0aa9d"
EVALUATION_COMMIT = "519f759b7fad88668f9ec5511dc631ccf3e81494"
EVALUATION_SUMMARY_SHA256 = "b7c9c9ac4607d1b59f606787f20e89550bcbd1fdb57c2cf9d17fee97e967345d"
TEST_PREDICTIONS_SHA256 = "c5ed095091f5574f261286f7a7d6f7687061233ecdd7320e3b567f8a7ba6e57a"

input_dir = Path("/content/cropguard_export_inputs")
if input_dir.exists():
    shutil.rmtree(input_dir)
input_dir.mkdir()

def download(url, destination, expected_sha256=None):
    urllib.request.urlretrieve(url, destination)
    actual = hashlib.sha256(Path(destination).read_bytes()).hexdigest()
    if expected_sha256 and actual != expected_sha256:
        raise RuntimeError(f"Hash mismatch for {destination}: {actual}")
    return actual

audit_zip = input_dir / "audit.zip"
download(
    f"https://raw.githubusercontent.com/attabeezy/crop-guard/{AUDIT_COMMIT}/cropguard-data-prep.zip",
    audit_zip,
    AUDIT_SHA256,
)
audit_dir = input_dir / "audit"
with zipfile.ZipFile(audit_zip) as archive:
    archive.extractall(audit_dir)

model_path = input_dir / "best_finetuned.keras"
download(
    "https://media.githubusercontent.com/media/attabeezy/crop-guard/"
    f"{MODEL_COMMIT}/results/training-artifacts/best_finetuned.keras",
    model_path,
    MODEL_SHA256,
)

evaluation_summary_path = input_dir / "evaluation_summary.json"
test_predictions_path = input_dir / "test_predictions.csv"
download(
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{EVALUATION_COMMIT}/results/evaluation/evaluation_summary.json",
    evaluation_summary_path,
    EVALUATION_SUMMARY_SHA256,
)
download(
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{EVALUATION_COMMIT}/results/evaluation/test_predictions.csv",
    test_predictions_path,
    TEST_PREDICTIONS_SHA256,
)
evaluation = json.loads(evaluation_summary_path.read_text())
if evaluation["selected_checkpoint"] != "best_finetuned.keras":
    raise RuntimeError("Pinned evaluation selected a different checkpoint")
temperature = float(evaluation["calibration"]["temperature"])
threshold = float(evaluation["test"]["confidence_threshold_locked_on_validation"])
print("Verified model, manifests, and evaluation decisions")
print("Temperature:", temperature, "Threshold:", threshold)

## 3. Resolve dataset paths and class order

In [ ]:
import kagglehub

DATASET_HANDLE = "nirmalsankalana/crop-pest-and-disease-detection"
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
test_predictions = pd.read_csv(test_predictions_path)
classes = sorted(test_predictions["label"].unique())
class_to_index = {label: index for index, label in enumerate(classes)}
assert len(classes) == 22

train_frame = pd.read_csv(audit_dir / "train.csv")
test_frame = pd.read_csv(audit_dir / "test.csv")
for frame in (train_frame, test_frame):
    frame["absolute_path"] = frame["path"].map(lambda value: str(dataset_root / value))
    frame["target"] = frame["label"].map(class_to_index)
    if frame["target"].isna().any():
        raise RuntimeError("Manifest contains an unknown class")
    if not frame["absolute_path"].map(lambda value: Path(value).is_file()).all():
        raise RuntimeError("Manifest path missing from downloaded dataset")

# Remove only the two TensorFlow-incompatible training files documented by notebook #3.
decode_failures_url = (
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{MODEL_COMMIT}/results/training-artifacts/tensorflow_decode_failures.csv"
)
with urllib.request.urlopen(decode_failures_url) as response:
    decode_failures = pd.read_csv(io.BytesIO(response.read()))
train_frame = train_frame[~train_frame["path"].isin(set(decode_failures["path"]))].reset_index(drop=True)

# Preserve the exact evaluated test order and verify manifest agreement.
test_frame = test_predictions[["path"]].merge(test_frame, on="path", how="left", validate="one_to_one")
if test_frame["label"].isna().any():
    raise RuntimeError("Evaluation predictions do not match the test manifest")
assert np.array_equal(test_frame["label"].to_numpy(), test_predictions["label"].to_numpy())
print(f"Representative pool: {len(train_frame):,}; parity test: {len(test_frame):,}")

## 4. Load the selected float checkpoint

In [ ]:
model = tf.keras.models.load_model(model_path, compile=False)
if model.input_shape != (None, IMAGE_SIZE, IMAGE_SIZE, 3):
    raise RuntimeError(f"Unexpected input shape: {model.input_shape}")
if model.output_shape[-1] != len(classes):
    raise RuntimeError(f"Unexpected output shape: {model.output_shape}")
print("Float model loaded:", model.name)

## 5. Build the representative dataset

The model includes EfficientNetV2 preprocessing and expects float pixels in `[0, 255]`. Representative examples are a deterministic class-stratified sample from training only.

In [ ]:
def load_float_image(path):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
    return tf.cast(image, tf.float32)

samples_per_class = int(np.ceil(REPRESENTATIVE_IMAGES / len(classes)))
representative_frame = pd.concat([
    group.sample(n=min(samples_per_class, len(group)), random_state=SEED)
    for _, group in train_frame.groupby("label", sort=True)
], ignore_index=True).head(REPRESENTATIVE_IMAGES)
if representative_frame["label"].nunique() != 22:
    raise RuntimeError("Representative sample does not cover every class")
print("Representative images:", len(representative_frame))

def representative_dataset():
    for path in representative_frame["absolute_path"]:
        image = load_float_image(path)
        yield [tf.expand_dims(image, axis=0)]

## 6. Convert to full integer TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_bytes = converter.convert()

export_dir = Path("/content/cropguard_int8_export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir()
tflite_path = export_dir / "model.tflite"
tflite_path.write_bytes(tflite_bytes)
print(f"INT8 model size: {tflite_path.stat().st_size / 1_000_000:.2f} MB")
print("SHA-256:", hashlib.sha256(tflite_bytes).hexdigest())

## 7. Inspect and enforce the deployment tensor contract

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_path), num_threads=4)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

if input_detail["dtype"] != np.uint8 or output_detail["dtype"] != np.uint8:
    raise RuntimeError(
        f"Model is not full uint8: input={input_detail['dtype']}, output={output_detail['dtype']}"
    )
if tuple(input_detail["shape"]) != (1, IMAGE_SIZE, IMAGE_SIZE, 3):
    raise RuntimeError(f"Unexpected TFLite input: {input_detail['shape']}")
if tuple(output_detail["shape"]) != (1, len(classes)):
    raise RuntimeError(f"Unexpected TFLite output: {output_detail['shape']}")

input_scale, input_zero = input_detail["quantization"]
output_scale, output_zero = output_detail["quantization"]
if input_scale <= 0 or output_scale <= 0:
    raise RuntimeError("Missing quantization parameters")
print("Input:", input_detail["shape"], input_detail["dtype"], input_detail["quantization"])
print("Output:", output_detail["shape"], output_detail["dtype"], output_detail["quantization"])

## 8. Run full held-out INT8 parity evaluation

In [ ]:
from tqdm.auto import tqdm

def run_tflite(path):
    image = load_float_image(path).numpy()[None, ...]
    quantized = np.clip(
        np.round(image / input_scale + input_zero), 0, 255
    ).astype(np.uint8)
    interpreter.set_tensor(input_detail["index"], quantized)
    interpreter.invoke()
    raw = interpreter.get_tensor(output_detail["index"])[0]
    probabilities = (raw.astype(np.float32) - output_zero) * output_scale
    probabilities = np.clip(probabilities, 0.0, 1.0)
    total = probabilities.sum()
    return probabilities / total if total > 0 else probabilities

int8_probabilities = np.stack([
    run_tflite(path) for path in tqdm(test_frame["absolute_path"], desc="INT8 test")
])
int8_predictions = int8_probabilities.argmax(axis=1)
truth = test_frame["target"].to_numpy(dtype=np.int32)
float_predictions = test_predictions["predicted_label"].map(class_to_index).to_numpy()
print("Completed INT8 inference:", int8_probabilities.shape)

## 9. Apply the locked confidence policy and enforce parity gates

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def temperature_scale(probabilities, value):
    logits = np.log(np.clip(probabilities, 1e-8, 1.0)) / value
    logits -= logits.max(axis=1, keepdims=True)
    exponentiated = np.exp(logits)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)

calibrated_int8 = temperature_scale(int8_probabilities, temperature)
int8_confidence = calibrated_int8.max(axis=1)
int8_accepted = int8_confidence >= threshold
int8_correct = int8_predictions == truth
float_accuracy = float(evaluation["test"]["accuracy"])
float_macro_f1 = float(evaluation["test"]["macro_f1"])
int8_accuracy = float(accuracy_score(truth, int8_predictions))
int8_macro_f1 = float(f1_score(truth, int8_predictions, average="macro"))
agreement = float((int8_predictions == float_predictions).mean())
float_accepted = test_predictions["accepted"].astype(bool).to_numpy()

parity_summary = {
    "float_accuracy": float_accuracy,
    "int8_accuracy": int8_accuracy,
    "accuracy_drop": float_accuracy - int8_accuracy,
    "float_macro_f1": float_macro_f1,
    "int8_macro_f1": int8_macro_f1,
    "macro_f1_drop": float_macro_f1 - int8_macro_f1,
    "prediction_agreement": agreement,
    "changed_predictions": int((int8_predictions != float_predictions).sum()),
    "acceptance_decision_agreement": float((int8_accepted == float_accepted).mean()),
    "int8_coverage": float(int8_accepted.mean()),
    "int8_accepted_accuracy": float(int8_correct[int8_accepted].mean())
        if int8_accepted.any() else None,
}
PARITY_LIMITS = {
    "maximum_accuracy_drop": 0.01,
    "maximum_macro_f1_drop": 0.015,
    "minimum_prediction_agreement": 0.98,
}
parity_passed = (
    parity_summary["accuracy_drop"] <= PARITY_LIMITS["maximum_accuracy_drop"]
    and parity_summary["macro_f1_drop"] <= PARITY_LIMITS["maximum_macro_f1_drop"]
    and agreement >= PARITY_LIMITS["minimum_prediction_agreement"]
)
parity_summary["status"] = "pass" if parity_passed else "fail"
display(parity_summary)
if not parity_passed:
    print("PARITY GATE FAILED. Package will be produced for diagnosis but must not ship.")

## 10. Measure host-runtime latency

This is a Colab CPU benchmark, not a phone-performance claim.

In [ ]:
import time

benchmark_image = load_float_image(test_frame.iloc[0]["absolute_path"]).numpy()[None, ...]
benchmark_input = np.clip(
    np.round(benchmark_image / input_scale + input_zero), 0, 255
).astype(np.uint8)
for _ in range(10):
    interpreter.set_tensor(input_detail["index"], benchmark_input)
    interpreter.invoke()

latencies_ms = []
for _ in range(100):
    interpreter.set_tensor(input_detail["index"], benchmark_input)
    started = time.perf_counter()
    interpreter.invoke()
    latencies_ms.append((time.perf_counter() - started) * 1000)
latency_summary = {
    "environment": "Google Colab CPU; not Android device latency",
    "threads": 4,
    "runs": len(latencies_ms),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
}
display(latency_summary)

## 11. Export Android assets and reproducibility reports

In [ ]:
DISPLAY_OVERRIDES = {
    "healthy": "Healthy",
    "fall_armyworm": "Fall armyworm damage",
    "grasshopper": "Grasshopper damage",
    "leaf_beetle": "Leaf beetle damage",
    "green_mite": "Green mite damage",
    "leaf_miner": "Leaf miner damage",
}
with (export_dir / "labels.txt").open("w", encoding="utf-8") as stream:
    for label in classes:
        crop, key = label.split("|", 1)
        display = DISPLAY_OVERRIDES.get(key, key.replace("_", " ").title())
        stream.write(f"{crop}|{key}|{crop.title()} — {display}\n")

tensor_contract = {
    "model_sha256": hashlib.sha256(tflite_bytes).hexdigest(),
    "model_size_bytes": len(tflite_bytes),
    "input": {
        "shape": input_detail["shape"].tolist(),
        "dtype": np.dtype(input_detail["dtype"]).name,
        "quantization_scale": float(input_scale),
        "quantization_zero_point": int(input_zero),
        "source_pixel_range": [0.0, 255.0],
        "resize": [IMAGE_SIZE, IMAGE_SIZE],
    },
    "output": {
        "shape": output_detail["shape"].tolist(),
        "dtype": np.dtype(output_detail["dtype"]).name,
        "quantization_scale": float(output_scale),
        "quantization_zero_point": int(output_zero),
        "class_order_file": "labels.txt",
    },
    "postprocessing": {
        "temperature": temperature,
        "confidence_threshold": threshold,
        "formula": "softmax(log(clip(probabilities,1e-8,1))/temperature)",
    },
}
(export_dir / "tensor_contract.json").write_text(json.dumps(tensor_contract, indent=2))

deployment_report = {
    "tensorflow_version": tf.__version__,
    "source_model_commit": MODEL_COMMIT,
    "source_model_sha256": MODEL_SHA256,
    "audit_commit": AUDIT_COMMIT,
    "evaluation_commit": EVALUATION_COMMIT,
    "representative_images": len(representative_frame),
    "representative_classes": int(representative_frame["label"].nunique()),
    "parity_limits": PARITY_LIMITS,
    "parity": parity_summary,
    "latency": latency_summary,
    "release_eligible": parity_passed,
}
(export_dir / "deployment_report.json").write_text(
    json.dumps(deployment_report, indent=2)
)

parity_predictions = test_predictions[["path", "label", "predicted_label"]].copy()
parity_predictions["int8_predicted_label"] = [classes[index] for index in int8_predictions]
parity_predictions["int8_confidence"] = int8_confidence
parity_predictions["int8_accepted"] = int8_accepted
parity_predictions["prediction_matches_float"] = int8_predictions == float_predictions
parity_predictions.to_csv(export_dir / "int8_parity_predictions.csv", index=False)

archive = shutil.make_archive("/content/cropguard-int8-export", "zip", export_dir)
print(f"Created {archive} ({Path(archive).stat().st_size / 1_000_000:.1f} MB)")
print("Archive SHA-256:", hashlib.sha256(Path(archive).read_bytes()).hexdigest())

In [ ]:
from google.colab import files
files.download("/content/cropguard-int8-export.zip")

## Release decision

Do not integrate `model.tflite` if `deployment_report.json` says `release_eligible: false`. If parity passes, provide the ZIP for inspection before the Android input preprocessing and post-processing code are updated to match `tensor_contract.json`.